In [355]:
import pygad
import numpy as np
import itertools

In [356]:
A = [7, 8, 18]
B = [1,2,2,4,5,5,5,9]
C = [1,2,2,2,2,3,4,5,5,7]
m = len(A)
n = len(B)
target = sorted(C)
print(f"A:{len(A)} B: {len(B)} C: {len(C)}")

A:3 B: 8 C: 10


In [357]:
def cumulative_sum(arr):
    return np.cumsum(arr).tolist()

In [358]:
def step_difference(arr):
    return [arr[0]] + [arr[i] - arr[i-1] for i in range(1, len(arr))]

In [391]:
def fitness_func(ga, solution, solution_idx) -> float:
    # chromosome encoding: [ u | ν ]

    mu = solution[:m]
    nu = solution[m:]


    # Check if valid permutations
    if len(set(mu)) != m or len(set(nu)) != n:
        return 0.0  # invalid permutation

    A_perm = [A[i] for i in mu]
    B_perm = [B[i] for i in nu]
    C_mod = list(target)

    k = len(C_mod)
    expected_k = m + n - 1

    # --- REPAIR SEQUENCE LOGIC ---
    if k < expected_k:
        # Rule 1: Add 0-length fragments to target C
        delta = expected_k - k
        C_mod.extend([0] * delta)

    elif k > expected_k:
        # Rule 2: Add 0-length fragments to the shorter sequence among A and B
        delta = k - expected_k
        if m <= n:
            A_perm.extend([0] * delta)
        else:
            B_perm.extend([0] * delta)

    A_cum = cumulative_sum(A_perm)
    B_cum = cumulative_sum(B_perm)

    merged = sorted(A_cum + B_cum)

    steps = step_difference(merged)

    steps_sorted = sorted(set(steps))



    error = sum(abs(s - t) for s, t in zip(steps_sorted, target))
    fitness = 1.0 / (1.0 + error)
    print(f"mu: {mu}; nu:{nu} -> fitness: {fitness}")
    return fitness

In [392]:
def get_perms(lst) -> []:
    indices = list(range(len(lst)))  # [0, 1, 2]

    # all permutations of indices
    perms_indices = list(itertools.permutations(indices))

    # convert tuples to lists (optional)
    perms_indices = [list(p) for p in perms_indices]
    return perms_indices

In [393]:
gene_space = []
for _ in range(m): gene_space.append(list(range(m)))
for _ in range(n):  gene_space.append(list(range(n)))
gene_space

[[0, 1, 2],
 [0, 1, 2],
 [0, 1, 2],
 [0, 1, 2, 3, 4, 5, 6, 7],
 [0, 1, 2, 3, 4, 5, 6, 7],
 [0, 1, 2, 3, 4, 5, 6, 7],
 [0, 1, 2, 3, 4, 5, 6, 7],
 [0, 1, 2, 3, 4, 5, 6, 7],
 [0, 1, 2, 3, 4, 5, 6, 7],
 [0, 1, 2, 3, 4, 5, 6, 7],
 [0, 1, 2, 3, 4, 5, 6, 7]]

In [398]:
ga = pygad.GA(
    num_generations=1000,
    num_parents_mating=50,
    gene_type=int,
    fitness_func=fitness_func,
    sol_per_pop= 100,
    num_genes=n+m,
    gene_space=gene_space,
    parent_selection_type="tournament",
    crossover_type="single_point",
    crossover_probability=0.75,
    mutation_type="random",
    mutation_probability=0.05,
    keep_parents=50,
    allow_duplicate_genes=True,

)

In [403]:
ga.run()

mu: [0 1 2]; nu:[4 3 7 0 2 6 1 5] -> fitness: 0.1
mu: [0 1 2]; nu:[4 6 0 7 2 3 1 5] -> fitness: 0.125
mu: [0 1 2]; nu:[4 3 0 7 2 6 1 5] -> fitness: 0.125
mu: [1 0 2]; nu:[4 3 7 0 2 6 1 5] -> fitness: 0.1
mu: [2 1 0]; nu:[4 3 7 0 2 6 1 5] -> fitness: 0.07142857142857142
mu: [0 1 2]; nu:[4 2 7 0 3 6 1 5] -> fitness: 0.07692307692307693
mu: [0 1 2]; nu:[4 3 0 7 2 6 1 5] -> fitness: 0.125
mu: [0 1 2]; nu:[4 6 7 0 2 3 1 5] -> fitness: 0.125
mu: [0 1 2]; nu:[4 3 0 7 2 6 1 5] -> fitness: 0.125
mu: [1 0 2]; nu:[4 3 7 0 2 6 1 5] -> fitness: 0.1
mu: [0 1 2]; nu:[4 6 7 0 2 3 1 5] -> fitness: 0.125
mu: [0 1 2]; nu:[4 6 7 0 2 3 1 5] -> fitness: 0.125
mu: [1 0 2]; nu:[4 6 0 7 2 3 1 5] -> fitness: 0.125
mu: [0 1 2]; nu:[4 6 0 7 3 2 1 5] -> fitness: 0.125
mu: [2 1 0]; nu:[4 6 0 7 2 3 1 5] -> fitness: 0.08333333333333333
mu: [1 0 2]; nu:[4 6 0 7 2 3 1 5] -> fitness: 0.125
mu: [1 0 2]; nu:[4 6 0 7 2 3 1 5] -> fitness: 0.125
mu: [2 1 0]; nu:[4 6 0 7 2 3 1 5] -> fitness: 0.08333333333333333
mu: [2 1 0]; n

In [404]:
solution, fitness, _ = ga.best_solution()

In [405]:
print("Best fitness:", fitness)

mu = solution[:m].astype(int)
nu = solution[m:].astype(int)

print("μ:", mu)
print("ν:", nu)

Best fitness: 0.125
μ: [0 1 2]
ν: [4 6 0 7 2 3 1 5]


In [402]:

A_perm = [A[i] for i in mu]
B_perm = [B[i] for i in nu]

A_cum = cumulative_sum(A_perm)
B_cum = cumulative_sum(B_perm)

merged = sorted(A_cum + B_cum)

steps = step_difference(merged)

steps_sorted = sorted(steps)
print(f"Is solution ok: {steps_sorted == target}")

Is solution ok: False
